# 🌿 YOLOv8 Leaf Detection — Dual T4 GPU (Kaggle)

**Features implemented:**
- ✅ Dual T4 GPU training via Ultralytics DDP + Accelerate `notebook_launcher`
- ✅ Google Drive dataset download & unzip
- ✅ Auto class-remapping → single `leaf` class
- ✅ 640 × 640 resolution, optimal batch size
- ✅ Preprocessing (mosaic, augmentation)
- ✅ Early stopping at patience = 10
- ✅ Multi-threaded workers for fast data loading
- ✅ Checkpoint / resume (never lose progress)
- ✅ Background / persistent session training
- ✅ Model info: parameters + FLOPs
- ✅ INT8 ONNX export (Kaggle / server inference)
- ✅ **NCNN INT8 export — recommended for Raspberry Pi 5** (2–3× faster than TFLite on Cortex-A76)
- ✅ TFLite INT8 export (secondary / fallback edge option)

> **Important:** Enable **2 × T4 GPUs** in *Notebook Settings → Accelerator* before running.

## 1 · Install / Upgrade Dependencies

In [ ]:
# Install / upgrade required packages
!pip install -q --upgrade ultralytics accelerate gdown onnx onnxruntime
!pip install -q onnxruntime-gpu  # GPU-accelerated ONNX runtime for INT8 inference
# ncnn: Ultralytics auto-installs the 'ncnn' Python bindings during export


## 2 · Environment & GPU Check

In [ ]:
import os, sys, shutil, yaml, json, time, threading, subprocess
import multiprocessing
import torch
from pathlib import Path

# ── GPU info ──────────────────────────────────────────────────────────────────
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
num_gpus = torch.cuda.device_count()
print(f"GPU count       : {num_gpus}")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    vram  = props.total_memory / 1e9
    print(f"  GPU {i}: {props.name}  VRAM={vram:.1f} GB")

# ── CPU / worker info ─────────────────────────────────────────────────────────
cpu_count = multiprocessing.cpu_count()
print(f"\nCPU logical cores: {cpu_count}")

# Optimal workers: min(cpu_count, 8) — avoids memory pressure
NUM_WORKERS = min(cpu_count, 8)
print(f"DataLoader workers: {NUM_WORKERS}")

## 3 · Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  USER CONFIGURATION — edit only this cell
# ═══════════════════════════════════════════════════════════════════════════════

# Google Drive share link (or direct file-id) to your zipped dataset
# Example: 'https://drive.google.com/file/d/FILE_ID/view?usp=sharing'
GDRIVE_URL = "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"

# Where to extract the dataset
DATASET_ROOT = Path("/kaggle/working/dataset")

# Output directory for training runs
RUNS_DIR = Path("/kaggle/working/runs")

# Project / experiment name (used for checkpointing)
PROJECT_NAME = "leaf_detect"
EXP_NAME     = "yolov8n_leaf_v1"

# Training hyper-parameters
MODEL_VARIANT  = "yolov8n.pt"   # nano — best speed/accuracy trade-off on T4
IMG_SIZE       = 640
EPOCHS         = 100
PATIENCE       = 10             # early-stopping patience
SEED           = 42

# Batch size strategy:
#   T4 has 16 GB VRAM.  With 2 × T4 via DDP the effective batch doubles.
#   YOLOv8n at 640 px fits ~32–64 images per GPU comfortably.
#   Set BATCH = -1 to let Ultralytics auto-detect the maximum safe batch.
BATCH = -1   # -1 = auto (recommended for dual-GPU Kaggle)

# Mixed-precision and optimization
AMP         = True   # Automatic Mixed Precision (FP16) — mandatory for T4
OPTIMIZER   = "AdamW"
LR0         = 0.01
WEIGHT_DECAY = 5e-4

# Single-class target label
TARGET_CLASS_NAME = "leaf"

# ═══════════════════════════════════════════════════════════════════════════════
print("Configuration loaded ✓")

## 4 · Download & Unzip Dataset from Google Drive

In [ ]:
import gdown, zipfile

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/kaggle/working/dataset.zip")

if not ZIP_PATH.exists():
    print("Downloading dataset from Google Drive …")
    gdown.download(GDRIVE_URL, str(ZIP_PATH), quiet=False, fuzzy=True)
else:
    print(f"Zip already exists: {ZIP_PATH}")

print("Unzipping …")
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(DATASET_ROOT)

print(f"Dataset extracted to: {DATASET_ROOT}")
# Show top-level structure
for p in sorted(DATASET_ROOT.rglob("*"))[:30]:
    print(" ", p.relative_to(DATASET_ROOT))

## 5 · Remap All Classes → Single `leaf` Class

This cell scans every YOLO `.txt` label file in the dataset and replaces
whichever class index is present with `0` (the single `leaf` class).

In [ ]:
def remap_labels_to_single_class(dataset_root: Path, class_name: str = "leaf") -> dict:
    """
    Overwrite every YOLO label file so that ALL class indices become 0.
    Returns a summary dict with counts and the new data.yaml path.
    """
    label_files = list(dataset_root.rglob("*.txt"))
    # Exclude data.yaml-companion note files that aren't label files
    label_files = [f for f in label_files if f.parent.name in ("labels", "train", "val", "test")
                   or any(part in ("labels",) for part in f.parts)]

    changed = 0
    for lf in label_files:
        lines = lf.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:          # class cx cy w h [conf]
                parts[0] = "0"           # force class → 0
                new_lines.append(" ".join(parts))
        if new_lines != lines:
            changed += 1
        lf.write_text("\n".join(new_lines) + "\n")

    print(f"Processed {len(label_files)} label files, {changed} updated.")
    return len(label_files)


def find_or_create_data_yaml(dataset_root: Path, class_name: str = "leaf") -> Path:
    """
    Find an existing data.yaml inside the extracted dataset, update it to
    use a single class, and return its path.  Creates one if none is found.
    """
    yaml_files = list(dataset_root.rglob("*.yaml")) + list(dataset_root.rglob("*.yml"))
    # Prefer files named 'data.yaml' or 'dataset.yaml'
    preferred = [f for f in yaml_files if f.stem in ("data", "dataset")]
    yaml_path = preferred[0] if preferred else (yaml_files[0] if yaml_files else None)

    if yaml_path:
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        print(f"Found existing YAML: {yaml_path}")
        print(f"  Original classes ({cfg.get('nc', '?')}): {cfg.get('names', [])}")
    else:
        print("No YAML found — generating one …")
        # Auto-detect train/val directories
        train_dir = next(dataset_root.rglob("images/train"), None) or \
                    next(dataset_root.rglob("train/images"), None) or \
                    dataset_root / "train"
        val_dir   = next(dataset_root.rglob("images/val"),   None) or \
                    next(dataset_root.rglob("val/images"),   None) or \
                    dataset_root / "val"
        cfg = {"path": str(dataset_root),
               "train": str(train_dir.relative_to(dataset_root)),
               "val":   str(val_dir.relative_to(dataset_root))}
        yaml_path = dataset_root / "data.yaml"

    # Overwrite class info
    cfg["nc"]    = 1
    cfg["names"] = [class_name]
    # Ensure absolute paths are preserved or made relative
    if "path" not in cfg:
        cfg["path"] = str(dataset_root)

    with open(yaml_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)

    print(f"Updated YAML saved → {yaml_path}")
    print(f"  nc=1, names=['{class_name}']")
    return yaml_path


# Run remapping
total_labels = remap_labels_to_single_class(DATASET_ROOT, TARGET_CLASS_NAME)
DATA_YAML    = find_or_create_data_yaml(DATASET_ROOT, TARGET_CLASS_NAME)
print(f"\ndata.yaml ready: {DATA_YAML}")

## 6 · Model Info: Parameters & FLOPs

In [ ]:
from ultralytics import YOLO
import torch

# Load model on CPU first (no CUDA operations before notebook_launcher!)
_info_model = YOLO(MODEL_VARIANT)

# ── Parameter count ───────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in _info_model.model.parameters())
trainable_params = sum(p.numel() for p in _info_model.model.parameters() if p.requires_grad)

print("═" * 55)
print(f"  Model          : {MODEL_VARIANT}")
print(f"  Total params   : {total_params:>15,}")
print(f"  Trainable      : {trainable_params:>15,}")
print("═" * 55)

# ── FLOPs via Ultralytics built-in info() ─────────────────────────────────────
# info() prints GFLOPs, layers, and parameter summary
print("\nUltralytics model summary:")
_info_model.info(verbose=True, imgsz=IMG_SIZE)

# ── Token / throughput estimate ───────────────────────────────────────────────
# YOLOv8 is not a generative model, so 'tokens' here means
# grid-cell predictions per image (i.e. the detection heads output).
# The three detection strides (8, 16, 32) divide IMG_SIZE to give the
# P3, P4, P5 grid dimensions, all derived from IMG_SIZE at runtime.
P3_GRID = IMG_SIZE // 8    # e.g. 640 // 8  = 80
P4_GRID = IMG_SIZE // 16   # e.g. 640 // 16 = 40
P5_GRID = IMG_SIZE // 32   # e.g. 640 // 32 = 20
GRID_PREDICTIONS = P3_GRID**2 + P4_GRID**2 + P5_GRID**2
print(f"\nDetection predictions per image (≈ 'tokens'): {GRID_PREDICTIONS:,}")
print(f"  P3 {P3_GRID}×{P3_GRID} + P4 {P4_GRID}×{P4_GRID} + P5 {P5_GRID}×{P5_GRID} = {GRID_PREDICTIONS:,} grid cells")

del _info_model  # free memory before CUDA launch

## 7 · Training Function (Dual-GPU via Accelerate + Ultralytics DDP)

> **Rule:** All CUDA ops must live *inside* this function — never in the
> global notebook scope — otherwise `notebook_launcher` will fail.

In [ ]:
def training_loop(
    data_yaml: str,
    model_variant: str,
    img_size: int,
    epochs: int,
    batch: int,
    patience: int,
    num_workers: int,
    project: str,
    name: str,
    lr0: float,
    weight_decay: float,
    optimizer: str,
    seed: int,
    amp: bool,
):
    """
    Full training loop — launched by notebook_launcher so that it runs
    across both T4 GPUs using PyTorch DDP under the hood.

    Ultralytics YOLO.train() accepts `device='0,1'` which internally
    spawns a DDP process group identical to what Accelerate would create,
    but with YOLO-specific optimizations (mosaic, augment, etc.).
    Wrapping it inside notebook_launcher ensures the notebook cell itself
    does not block and CUDA is initialised safely inside child processes.
    """
    import os, torch
    from accelerate import Accelerator
    from ultralytics import YOLO
    from pathlib import Path

    # ── Accelerator (handles rank / device placement) ─────────────────────────
    accelerator = Accelerator(mixed_precision="fp16" if amp else "no")
    rank        = accelerator.process_index   # 0 = main GPU
    device_id   = rank                        # GPU 0 or GPU 1

    accelerator.print("\n" + "═" * 60)
    accelerator.print(f"  Dual-GPU YOLOv8 Leaf Detection")
    accelerator.print(f"  Rank {rank} / {accelerator.num_processes} on {torch.cuda.get_device_name(device_id)}")
    accelerator.print("═" * 60)

    # ── Resolve checkpoint: resume from last run if available ─────────────────
    run_dir     = Path(project) / name
    weights_dir = run_dir / "weights"
    last_ckpt   = weights_dir / "last.pt"
    best_ckpt   = weights_dir / "best.pt"

    if last_ckpt.exists():
        model_path = str(last_ckpt)
        accelerator.print(f"  ✅ Resuming from checkpoint: {last_ckpt}")
        resume_flag = True
    else:
        model_path  = model_variant
        resume_flag = False
        accelerator.print(f"  🚀 Starting fresh training with: {model_variant}")

    # ── Load YOLO model ───────────────────────────────────────────────────────
    # NOTE: YOLO() internally calls torch.cuda — hence this must be inside
    #       the training_loop function, never at notebook cell level.
    model = YOLO(model_path)

    # ── Train ─────────────────────────────────────────────────────────────────
    # device='0,1' → Ultralytics DDP across both GPUs
    # resume=True  → restore training state from last.pt
    results = model.train(
        data        = data_yaml,
        epochs      = epochs,
        imgsz       = img_size,
        batch       = batch,           # -1 = auto-batch
        device      = "0,1",           # both T4s
        workers     = num_workers,
        amp         = amp,
        optimizer   = optimizer,
        lr0         = lr0,
        weight_decay = weight_decay,
        patience    = patience,        # early stopping
        project     = project,
        name        = name,
        exist_ok    = True,            # overwrite if re-running
        resume      = resume_flag,
        seed        = seed,
        # ── Preprocessing / augmentation ──────────────────────────────────────
        hsv_h       = 0.015,   # hue jitter
        hsv_s       = 0.7,     # saturation
        hsv_v       = 0.4,     # brightness
        degrees     = 10.0,    # rotation
        translate   = 0.1,     # translation
        scale       = 0.5,     # scale
        flipud      = 0.5,     # vertical flip
        fliplr      = 0.5,     # horizontal flip
        mosaic      = 1.0,     # mosaic (4-image) augmentation
        mixup       = 0.1,     # mixup
        copy_paste  = 0.1,     # copy-paste augmentation
        # ── Misc ──────────────────────────────────────────────────────────────
        plots       = True,    # save training plots
        save_period = 5,       # checkpoint every 5 epochs
        verbose     = (rank == 0),  # only main process prints
    )

    # ── Only main process does post-training steps ────────────────────────────
    accelerator.wait_for_everyone()

    if accelerator.is_main_process:
        accelerator.print("\n✅ Training complete!")
        accelerator.print(f"   Best weights : {best_ckpt}")
        accelerator.print(f"   Last weights : {last_ckpt}")

        # ── INT8 Export ───────────────────────────────────────────────────────
        if best_ckpt.exists():
            accelerator.print("\nExporting best model \u2026")
            best_model = YOLO(str(best_ckpt))

            # ── NCNN INT8 — PRIMARY for Raspberry Pi 5 ────────────────────
            # RPi 5 uses Cortex-A76 (ARM64). NCNN uses ARM NEON intrinsics
            # and can offload to the VideoCore VII GPU via Vulkan — typically
            # 2-3x faster than TFLite on the same hardware.
            try:
                ncnn_path = best_model.export(
                    format = "ncnn",
                    imgsz  = img_size,
                    int8   = True,   # INT8 quantisation via NCNN calibration
                    half   = False,  # INT8 supersedes FP16; use one or the other
                )
                accelerator.print(f"   ✅ NCNN INT8 → {ncnn_path}  ← deploy this on Raspberry Pi 5")
            except Exception as e:
                accelerator.print(f"   NCNN export failed ({e})")

            # ── ONNX INT8 — server / desktop fallback ─────────────────
            try:
                onnx_path = best_model.export(
                    format   = "onnx",
                    imgsz    = img_size,
                    int8     = True,
                    dynamic  = False,
                    simplify = True,
                )
                accelerator.print(f"   ONNX INT8 → {onnx_path}")
            except Exception as e:
                accelerator.print(f"   ONNX export failed ({e})")

            # ── TFLite INT8 — secondary edge option ──────────────────
            # Slower than NCNN on RPi 5 but more portable across Android/MCU.
            try:
                tflite_path = best_model.export(
                    format = "tflite",
                    imgsz  = img_size,
                    int8   = True,
                )
                accelerator.print(f"   TFLite INT8 → {tflite_path}  (fallback)")
            except Exception as e:
                accelerator.print(f"   TFLite export skipped ({e})")

        # ── Final model summary ───────────────────────────────────────────────
        accelerator.print("\n" + "═" * 60)
        accelerator.print("  Training Summary")
        accelerator.print("═" * 60)
        try:
            accelerator.print(f"  mAP50-95 (final) : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
            accelerator.print(f"  mAP50    (final) : {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
            accelerator.print(f"  Precision        : {results.results_dict.get('metrics/precision(B)', 'N/A')}")
            accelerator.print(f"  Recall           : {results.results_dict.get('metrics/recall(B)', 'N/A')}")
        except Exception:
            pass
        accelerator.print("═" * 60)

## 8 · Enable Session Persistence (Background Training)

Kaggle notebooks time-out when Chrome closes.  
This cell writes a small shell wrapper so that the training process
survives a browser disconnect via `nohup` + process lock.

In [ ]:
import subprocess, os
from pathlib import Path

KEEP_ALIVE_SCRIPT = Path("/kaggle/working/keep_alive.sh")
KEEP_ALIVE_SCRIPT.write_text("""\
#!/bin/bash
# Ping the Kaggle session endpoint every 30 s to prevent idle timeout.
while true; do
    curl -s --max-time 5 http://localhost:8080/api/kernels > /dev/null 2>&1 || true
    sleep 30
done
""")
KEEP_ALIVE_SCRIPT.chmod(0o755)

# Start keep-alive in the background
proc = subprocess.Popen(
    ["nohup", "bash", str(KEEP_ALIVE_SCRIPT)],
    stdout=open("/kaggle/working/keep_alive.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,   # detach from notebook process group
)
print(f"Keep-alive daemon started (PID {proc.pid}) ✓")
print("Training will continue even if you close this browser tab.")
print("Check /kaggle/working/keep_alive.log for status.")

## 9 · Launch Dual-GPU Training via `notebook_launcher`

`num_processes=2` → one process per T4 GPU.  
All CUDA initialisation happens *inside* `training_loop`, never before.

In [ ]:
from accelerate import notebook_launcher

# Bundle all arguments as a tuple — matches the training_loop signature
training_args = (
    str(DATA_YAML),    # data_yaml
    MODEL_VARIANT,     # model_variant
    IMG_SIZE,          # img_size
    EPOCHS,            # epochs
    BATCH,             # batch  (-1 = auto)
    PATIENCE,          # patience
    NUM_WORKERS,       # num_workers
    str(RUNS_DIR / PROJECT_NAME),  # project
    EXP_NAME,          # name
    LR0,               # lr0
    WEIGHT_DECAY,      # weight_decay
    OPTIMIZER,         # optimizer
    SEED,              # seed
    AMP,               # amp
)

print("🚀 Launching training across 2 T4 GPUs …")
print(f"   Data YAML  : {DATA_YAML}")
print(f"   Model      : {MODEL_VARIANT}")
print(f"   Image size : {IMG_SIZE}")
print(f"   Epochs     : {EPOCHS}  (early-stop patience={PATIENCE})")
print(f"   Batch      : {'auto' if BATCH == -1 else BATCH}")
print(f"   Workers    : {NUM_WORKERS}")
print(f"   AMP (FP16) : {AMP}")
print()

notebook_launcher(
    training_loop,
    training_args,
    num_processes=2,   # 2 × T4
    use_port="29500",  # explicit port avoids conflicts on re-runs
)

## 10 · Post-Training: Model Info & INT8 Validation

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import os

best_pt = RUNS_DIR / PROJECT_NAME / EXP_NAME / "weights" / "best.pt"

if best_pt.exists():
    print(f"Loading best checkpoint: {best_pt}")
    final_model = YOLO(str(best_pt))
    final_model.info(verbose=True, imgsz=IMG_SIZE)

    # ── List exported artefacts ───────────────────────────────────────────────
    run_dir = RUNS_DIR / PROJECT_NAME / EXP_NAME
    print("\nGenerated artefacts:")
    for f in sorted(run_dir.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / 1e6
            print(f"  {f.relative_to(run_dir):<55} {size_mb:>6.2f} MB")
else:
    print(f"⚠️  best.pt not found at {best_pt} — training may still be running.")

## 11 · Quick INT8 Inference Test

In [ ]:
import glob
from pathlib import Path
from ultralytics import YOLO

run_dir  = RUNS_DIR / PROJECT_NAME / EXP_NAME
onnx_int8 = list(run_dir.glob("*.onnx"))

if onnx_int8:
    onnx_path = str(onnx_int8[0])
    print(f"Testing INT8 ONNX model: {onnx_path}")
    int8_model = YOLO(onnx_path)

    # Find a sample validation image
    val_images = list(DATASET_ROOT.rglob("images/val")) or \
                 list(DATASET_ROOT.rglob("val/images"))
    sample_imgs = []
    if val_images:
        sample_imgs = list(val_images[0].glob("*.jpg"))[:5] + \
                      list(val_images[0].glob("*.png"))[:5]

    if sample_imgs:
        results = int8_model.predict(
            source     = sample_imgs[:3],
            imgsz      = IMG_SIZE,
            conf       = 0.25,
            save       = True,
            project    = str(RUNS_DIR / "inference_int8"),
            name       = "test",
            exist_ok   = True,
        )
        print(f"INT8 inference results saved to {RUNS_DIR / 'inference_int8' / 'test'}")
        for r in results:
            print(f"  Detected {len(r.boxes)} leaf instance(s) in {Path(r.path).name}")
    else:
        print("No sample images found for inference test.")
else:
    print("No INT8 ONNX file found. Run training first.")

## 12 · TFLite vs NCNN — Which Is Better for Raspberry Pi 5?

| Criterion | TFLite | **NCNN** |
|---|---|---|
| **Primary CPU target** | ARM (generic) | **ARM NEON** intrinsics (Cortex-A55/A72/A76) |
| **RPi 5 GPU (VideoCore VII)** | ❌ No Vulkan backend | ✅ **Vulkan** — offloads layers to GPU |
| **Inference speed on RPi 5** | Baseline | **2–3× faster** |
| **Runtime dependencies** | TensorFlow Lite runtime | **Zero** (self-contained C++ lib) |
| **INT8 support** | ✅ | ✅ |
| **FP16 support** | ✅ | ✅ |
| **Model size** | ~same | ~same (slightly smaller) |
| **Ultralytics export** | `format='tflite'` | `format='ncnn'` |
| **Best use-case** | Android / MCU / Google Coral | **Raspberry Pi / ARM SBC** |

### ✅ Verdict: Use **NCNN** on Raspberry Pi 5

> The Raspberry Pi 5 uses a **Broadcom BCM2712** with four **Cortex-A76** cores.
> NCNN was built by Tencent specifically for ARM devices — it compiles with ARM NEON
> SIMD instructions and can optionally accelerate layers on the Pi 5’s
> **VideoCore VII GPU via Vulkan**, something TFLite cannot do on this hardware.
> In real-world YOLOv8 benchmarks on Cortex-A76 boards, NCNN achieves **∼2–3×**
> higher throughput than TFLite at the same INT8 precision level.
> TFLite remains the better choice for Android devices, Coral Edge TPU, or
> microcontrollers where NCNN is not available.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 12a · Standalone NCNN INT8 Export (run after training completes)
# ─────────────────────────────────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

best_pt = RUNS_DIR / PROJECT_NAME / EXP_NAME / 'weights' / 'best.pt'

if not best_pt.exists():
    raise FileNotFoundError(f'best.pt not found at {best_pt} — run training first')

export_model = YOLO(str(best_pt))

# ── NCNN INT8 (Raspberry Pi 5 primary) ─────────────────────────────
print('Exporting to NCNN INT8 (Raspberry Pi 5) …')
ncnn_path = export_model.export(
    format = 'ncnn',
    imgsz  = IMG_SIZE,
    int8   = True,
    half   = False,
)
print(f'NCNN INT8 model → {ncnn_path}')
ncnn_dir = Path(str(ncnn_path))
if ncnn_dir.is_dir():
    for f in sorted(ncnn_dir.iterdir()):
        print(f'  {f.name:<40} {f.stat().st_size / 1e3:.1f} KB')

# ── TFLite INT8 (secondary / fallback) ────────────────────────────
print('\nExporting to TFLite INT8 (fallback) …')
try:
    tflite_path = export_model.export(
        format = 'tflite',
        imgsz  = IMG_SIZE,
        int8   = True,
    )
    print(f'TFLite INT8 model → {tflite_path}')
except Exception as e:
    print(f'TFLite export skipped: {e}')

print('\n✅ Export complete!')
print('Copy the NCNN directory to Raspberry Pi 5 for deployment.')

## 13 · Raspberry Pi 5 Deployment Guide

### Why NCNN beats TFLite on Raspberry Pi 5
- **ARM NEON** — NCNN uses hand-written NEON SIMD kernels; TFLite uses generic ARM code
- **Vulkan GPU** — RPi 5’s VideoCore VII supports Vulkan; only NCNN can use it
- **Zero dependencies** — NCNN is self-contained; TFLite needs the TF Lite runtime
- **Smaller RAM footprint** — NCNN’s C++ runtime uses less memory than TFLite

### Step 1 — Download NCNN model from Kaggle
```python
# In Kaggle notebook output panel, download the folder:
# runs/leaf_detect/yolov8n_leaf_v1/weights/best_ncnn_model/
# It contains: model.ncnn.bin, model.ncnn.param, metadata.yaml
```

### Step 2 — Install Ultralytics on Raspberry Pi 5
```bash
# On Raspberry Pi 5 (64-bit Raspberry Pi OS)
pip install ultralytics
# Ultralytics automatically installs ncnn Python bindings
```

### Step 3 — Enable Vulkan acceleration (optional but recommended)
```bash
# Install Vulkan driver for VideoCore VII
sudo apt install -y mesa-vulkan-drivers
# Tell NCNN to use Vulkan
export NCNN_VULKAN=1
```

### Step 4 — Run inference

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 13a · Raspberry Pi 5 Inference Script
# Run this code ON YOUR RASPBERRY PI 5 (not on Kaggle)
# ─────────────────────────────────────────────────────────────────────────
#
# Prerequisites (on RPi 5):  pip install ultralytics
# Optional Vulkan:           export NCNN_VULKAN=1
# ─────────────────────────────────────────────────────────────────────────

# ---- copy everything below this line to rpi5_leaf_detect.py on your Pi ----

# import os, time
# from ultralytics import YOLO
#
# # Optional: enable Vulkan GPU (VideoCore VII on RPi 5)
# os.environ['NCNN_VULKAN'] = '1'
#
# NCNN_MODEL = '/home/pi/leaf_detect/best_ncnn_model'  # adjust to your path
# IMAGE      = '/home/pi/test_image.jpg'
# IMG_SIZE   = 640
# CONF       = 0.25
#
# model = YOLO(NCNN_MODEL)
#
# # Warm-up (first run compiles kernels)
# model.predict(IMAGE, imgsz=IMG_SIZE, conf=CONF, verbose=False)
#
# # Timed inference
# t0 = time.perf_counter()
# results = model.predict(IMAGE, imgsz=IMG_SIZE, conf=CONF, save=True)
# ms = (time.perf_counter() - t0) * 1000
#
# for r in results:
#     print(f'Detected {len(r.boxes)} leaf instance(s)  |  {ms:.1f} ms')
#
# # --- benchmark: compare NCNN vs TFLite ---
# # tfl = YOLO('/home/pi/leaf_detect/best_float32.tflite')
# # t1 = time.perf_counter()
# # tfl.predict(IMAGE, imgsz=IMG_SIZE, conf=CONF)
# # print(f'TFLite: {(time.perf_counter()-t1)*1000:.1f} ms')

# ---- end of RPi 5 script ----

print('Raspberry Pi 5 inference script ready.')
print('Copy the commented code above to rpi5_leaf_detect.py on your Pi 5.')
print()
print('Format comparison for Raspberry Pi 5:')
rows = [
    ('NCNN INT8',   'fastest  — ARM NEON + optional Vulkan GPU', '← RECOMMENDED'),
    ('TFLite INT8', 'slower   — no Vulkan on RPi 5',            '(fallback)'),
    ('ONNX INT8',   'slowest  — generic CPU via onnxruntime',   '(desktop only)'),
]
for fmt, speed, note in rows:
    print(f'  {fmt:<14} {speed:<44} {note}')

---
## Notes & Tips

| Topic | Detail |
|---|---|
| **GPU Selection** | Kaggle → *Notebook Settings* → *Accelerator* → **2 × T4 GPU** |
| **Batch auto-detection** | `batch=-1` lets Ultralytics measure VRAM and pick the largest safe batch |
| **Resume after timeout** | Re-run cell 9 — the code detects `last.pt` and resumes automatically |
| **Session keepalive** | Cell 8 starts a background ping loop; enable *'Keep notebook running'* in Kaggle settings too |
| **RPi 5 format** | **NCNN INT8** — 2–3× faster than TFLite on Cortex-A76; Vulkan GPU support (VideoCore VII) |
| **INT8 ONNX** | Saved next to `best.pt` in `runs/leaf_detect/yolov8n_leaf_v1/` |
| **Single-class dataset** | All original class IDs in every `.txt` label are remapped to `0 (leaf)` |
| **Early stopping** | Training stops automatically if mAP doesn't improve for 10 consecutive epochs |
| **Workers** | Set to `min(cpu_count, 8)` for balanced I/O vs memory pressure |